# COMP9444 Group Project Notebook
## Fruit and Vegetable Freshness Assessment

This notebook aggregates the complete project workflow and the selected experiments used in the final report. The primary task is binary classification of **fresh** versus **rotten** produce using frozen ImageNet CNN features and classical classifiers.

The notebook covers dataset verification, exploratory analysis, the ResNet-18 baseline, DenseNet-201 and ResNeXt-101 comparisons, classifier analysis, PCA and augmentation ablations, error analysis, and the final test evaluation. Full result tables remain in `results/tables/` for traceability.

## 1. Project objectives

1. Establish a reproducible baseline.
2. Compare DenseNet-201 and ResNeXt-101 as frozen feature extractors.
3. Compare SVM, LDA, and Bagging as classifiers for high-dimensional CNN features.
4. Measure the effects of PCA and training augmentation.
5. Select the final configuration using validation Macro-F1 and evaluate the test set only once.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from IPython.display import display
except ImportError:
    display = print

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'results').exists() and (PROJECT_ROOT.parent / 'results').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

RESULTS = PROJECT_ROOT / 'results'
TABLES = RESULTS / 'tables'
FIGURES = RESULTS / 'figures'
SEED = 42
plt.rcParams['figure.dpi'] = 120
print('Project root:', PROJECT_ROOT)
print('Seed:', SEED)

## 2. Dataset and exploratory analysis

The dataset contains 12,000 images from 20 original classes: five fruits and five vegetables, each with fresh and rotten variants. The modelling task remaps the original labels to the binary classes `fresh` and `rotten`. The official split is stratified and fixed with seed 42.

In [ ]:
split_summary = pd.read_csv(TABLES / 'split_summary.csv')
format_summary = pd.read_csv(TABLES / 'format_size_summary.csv')
image_stats = pd.read_csv(TABLES / 'image_stats.csv')
display(split_summary)
display(format_summary)
display(image_stats.head())
print('Total images:', int(split_summary.loc[split_summary['label'].eq('ALL'), 'total'].iloc[0]))
print('Readable images:', int(format_summary['readable_images'].iloc[0]))
print('Corrupt images:', int(format_summary['corrupt_images'].iloc[0]))

In [ ]:
class_rows = split_summary[~split_summary['label'].eq('ALL')].copy()
class_rows['freshness'] = class_rows['label'].str.startswith('fresh').map({True: 'fresh', False: 'rotten'})
class_totals = class_rows.groupby('freshness')[['train', 'val', 'test', 'total']].sum()
class_totals.plot(kind='bar', figsize=(7, 4), color=['#4C78A8', '#F58518', '#54A24B', '#B279A2'])
plt.title('Binary class distribution across splits')
plt.ylabel('Number of images')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 3. Common experimental protocol

All experiments use 224 x 224 RGB images and ImageNet normalization. Features are extracted from frozen ImageNet-pretrained backbones. When PCA is used, `StandardScaler` and PCA are fitted on training features only and then applied to validation or test features. Validation Macro-F1 is the primary selection metric.

The main controlled comparison uses the shared Stage 4 augmented feature case so that DenseNet-201, ResNeXt-101, Fusion, and all classifiers use the same data convention. The DenseNet augmentation study is reported separately as an ablation.

In [ ]:
feature_checks = {
    'DenseNet no_aug train': RESULTS / 'features/no_aug/densenet201_train_features.npy',
    'DenseNet no_aug val': RESULTS / 'features/no_aug/densenet201_val_features.npy',
    'DenseNet aug train': RESULTS / 'features/aug/densenet201_train_features.npy',
    'DenseNet aug val': RESULTS / 'features/aug/densenet201_val_features.npy',
    'ResNet-18 baseline train': RESULTS / 'features/resnet18_baseline/resnet18_train_features.npy',
    'ResNet-18 baseline val': RESULTS / 'features/resnet18_baseline/resnet18_val_features.npy',
}
for name, path in feature_checks.items():
    if path.exists():
        print(f'{name}: {np.load(path, mmap_mode="r").shape}')
    else:
        print(f'{name}: not available locally')

## 4. Reproducible project commands

The following commands are the repository entry points. Feature extraction and full classifier evaluation are intentionally not executed automatically in this notebook because they can require substantial time and local dataset access.

In [ ]:
commands = [
    'python src/preprocessing/download_data.py',
    'python src/preprocessing/check_images.py --data-dir <dataset-path>',
    'python src/preprocessing/make_split.py --data-dir <dataset-path>',
    'python src/feature_extraction/extract_densenet201_official.py --data-dir <dataset-path>',
    'python src/feature_extraction/extract_resnext101.py --data-dir <dataset-path>',
    'python src/classification/run_pca_classifiers.py --feature-case aug --classifiers svm lda bagging',
    'python src/feature_extraction/extract_resnet18_baseline.py --data-dir <dataset-path>',
    'python src/evaluation/analyze_resnet18_baseline.py',
    'python src/classification/run_pca_classifiers.py --feature-case no_aug --run-name final_densenet_svm_no_aug_pca95 --evaluate-test --test-case B --test-classifier svm',
]
print('\n'.join(commands))

## 5. ResNet-18 baseline

The controlled transfer-learning baseline uses a frozen ImageNet-pretrained ResNet-18, StandardScaler, PCA retaining 95% variance, and an RBF SVM. It uses no augmentation and does not load test features during baseline selection.

In [ ]:
baseline_metrics = pd.read_csv(TABLES / 'resnet18_baseline_metrics_val.csv')
baseline_errors = pd.read_csv(TABLES / 'resnet18_baseline_val_errors.csv')
display(baseline_metrics)
print('Baseline validation errors:', len(baseline_errors))

The baseline validation result is Accuracy 0.9761 and Macro-F1 0.9761. This provides a reference point for the frozen DenseNet-201 and ResNeXt-101 feature pipelines.

In [ ]:
baseline_matrix_path = FIGURES / 'resnet18_baseline_confusion_val.png'
if baseline_matrix_path.exists():
    from PIL import Image
    display(Image.open(baseline_matrix_path))

## 6. DenseNet-201 and ResNeXt-101 backbone comparison

The controlled comparison uses the same augmented feature case and SVM classifier. DenseNet-201 without PCA reaches validation Macro-F1 0.9783. ResNeXt-101 with PCA reaches 0.9528, while its no-PCA result is 0.9494. The result supports DenseNet-201 as the stronger representation for this task under the current protocol.

In [ ]:
classifier_results = pd.read_csv(TABLES / 'validation_classifier_comparison.csv')
svm_backbones = classifier_results[
    (classifier_results['classifier'].eq('svm')) &
    (classifier_results['case'].isin(['A', 'B', 'C', 'D', 'E', 'F']))
].copy()
display(svm_backbones[['case', 'case_name', 'accuracy', 'macro_f1', 'weighted_f1', 'seconds']])

In [ ]:
plot_rows = classifier_results[(classifier_results['classifier'].eq('svm')) & (classifier_results['case'].isin(['A', 'B', 'C', 'D']))].copy()
labels = plot_rows['case_name'].str.replace('\n', ' ', regex=False)
plt.figure(figsize=(8, 4))
plt.bar(labels, plot_rows['macro_f1'], color=['#4C78A8', '#72B7B2', '#F58518', '#E45756'])
plt.ylim(0.90, 1.00)
plt.ylabel('Validation Macro-F1')
plt.title('DenseNet-201 and ResNeXt-101 comparison')
plt.xticks(rotation=20, ha='right')
plt.tight_layout()
plt.show()

## 7. Classifier comparison

On the DenseNet-201 feature case, SVM is the best classifier by Macro-F1 (0.9783), followed closely by LDA (0.9772). Bagging is substantially weaker (0.9405) and takes much longer to run. This supports using SVM as the primary classifier for high-dimensional CNN features.

In [ ]:
densenet_classifiers = classifier_results[classifier_results['case'].eq('A')].copy()
display(densenet_classifiers[['classifier_name', 'accuracy', 'macro_f1', 'weighted_f1', 'seconds']])
plt.figure(figsize=(6, 4))
plt.bar(densenet_classifiers['classifier_name'], densenet_classifiers['macro_f1'], color=['#4C78A8', '#F58518', '#54A24B'])
plt.ylim(0.90, 1.00)
plt.ylabel('Validation Macro-F1')
plt.title('DenseNet-201 classifier comparison')
plt.tight_layout()
plt.show()

## 8. PCA and augmentation ablations

PCA is not universally beneficial. It slightly reduces DenseNet-201 SVM Macro-F1 by 0.0011, but improves ResNeXt-101 SVM Macro-F1 by 0.0033. For DenseNet-201, the no-augmentation SVM setting is stronger than the augmented setting: Macro-F1 0.9878 versus 0.9772. These results are treated as ablations rather than assuming that PCA or augmentation must improve performance.

In [ ]:
pca_effect = pd.read_csv(TABLES / 'validation_no_pca_vs_pca95_comparison.csv')
augmentation_effect = pd.read_csv(TABLES / 'densenet201_augmentation_metrics_val.csv')
display(pca_effect)
display(augmentation_effect[['case', 'classifier', 'pca_components', 'accuracy', 'macro_f1', 'total_seconds']])

In [ ]:
aug_svm = augmentation_effect[augmentation_effect['classifier'].eq('svm')].copy()
plt.figure(figsize=(6, 4))
plt.bar(aug_svm['case'], aug_svm['macro_f1'], color=['#4C78A8', '#E45756'])
plt.ylim(0.95, 1.00)
plt.ylabel('Validation Macro-F1')
plt.title('DenseNet-201 augmentation ablation')
plt.tight_layout()
plt.show()

## 9. Final test evaluation

The final configuration was selected using validation Macro-F1 and evaluated once on the held-out test split: DenseNet-201, no augmentation, PCA retaining 95% variance, and RBF SVM. The test result is not used for further selection.

In [ ]:
final_test = pd.read_csv(TABLES / 'stage4_final_densenet_svm_no_aug_pca95_FINAL_test_metrics.csv')
final_confusion = pd.read_csv(TABLES / 'stage4_final_densenet_svm_no_aug_pca95_FINAL_test_confusion_matrix.csv', index_col=0)
display(final_test)
display(final_confusion)
print('Test sample count:', int(final_confusion.to_numpy().sum()))

The selected configuration achieves test Accuracy 0.9828, Macro Precision 0.9829, Macro Recall 0.9827, Macro-F1 0.9828, and Weighted-F1 0.9828 on 1,800 test images.

## 10. Error analysis and limitations

The DenseNet-201 no-augmentation SVM validation analysis contains 22 errors out of 1,801 validation images. Typical difficult cases include mild surface spots, local discoloration, lighting variation, watermarks, complex backgrounds, and images that appear visually ambiguous relative to their labels.

Important limitations are the binary label remapping, the use of frozen ImageNet features, the fixed classifier settings, and the absence of an external test dataset. The validation set is used for model selection and the test set is reserved for the final reported evaluation.

## 11. Final conclusions

DenseNet-201 is the strongest backbone in the controlled comparison, and RBF SVM is the most effective classifier for the extracted CNN features. PCA can reduce dimensionality and runtime, but its effect depends on the backbone. Training augmentation is not automatically beneficial for this dataset; the no-augmentation DenseNet setting achieved the best validation result in the ablation. The final test result confirms that the selected pipeline generalizes well to the held-out split.